In [1]:
import sys
import os
venv_path = os.path.join(os.getcwd(), '.venv', 'Lib', 'site-packages')
if os.path.exists(venv_path) and venv_path not in sys.path:
    sys.path.insert(0, venv_path)

if os.name == 'nt':
    os.add_dll_directory(r'C:\Program Files\SuperTuxKart 1.5')

import multiprocessing as mp
import numpy as np

import torch
import torch.optim as optim
import torch.nn.functional as F
from actor import ActorNetwork
from critic import CriticNetwork
from env_worker import SingleInstanceHD


In [2]:
print('Inference worker ready.')


Inference worker ready.


In [3]:
actor_net  = ActorNetwork(state_dim=52)
critic_net = CriticNetwork(state_dim=52)

actor_net.load_state_dict(torch.load('best_actor.pth', map_location='cpu'))
critic_net.load_state_dict(torch.load('best_critic.pth', map_location='cpu'))

actor_net.eval()
critic_net.eval()
print('Model ready.')


Model ready.


In [4]:
def main():
    try:
        EpochLimit  = 1
        MAX_STEPS   = 1500  # Allows the kart plenty of time to complete the full lap
        FRAME_DELAY = 0.04  # 0.02 = ~50 FPS (normal real-time speed). Increase to 0.04 for slow-motion.

        for episode in range(EpochLimit):
            total_episode_reward = 0.0
            ProcessList          = []
            ConList              = []
            for i in range(1):
                ParentCon, ChildCon = mp.Pipe()

                process = mp.Process(target=SingleInstanceHD, args=(i, ChildCon, FRAME_DELAY))
                ProcessList.append(process)
                ConList.append(ParentCon)
                process.start()

            BatchStates = []
            BatchDones  = []

            for con in ConList:
                np_obs, reward, RaceDone = con.recv()
                BatchStates.append(np_obs)
                BatchDones.append(bool(RaceDone))

            print(f'=== Starting Visual Run at Normal Real-Time Speed (50 FPS) ===')

            for step in range(MAX_STEPS):
                state_tensor = torch.FloatTensor(np.array(BatchStates))

                if torch.isnan(state_tensor).any():
                    print('NaN detected in engine observations! Terminating episode.')
                    break

                with torch.no_grad():
                    action_dist    = actor_net(state_tensor)
                    sampled_action = action_dist.mean

                MemoryActions = []
                for i in range(len(ConList)):
                    steer_val = torch.clamp(sampled_action[i, 0], min=-1.0, max=1.0).item()
                    accel_val = torch.clamp(sampled_action[i, 1], min=0.0,  max=1.0).item()
                    BrakeVal  = torch.clamp(sampled_action[i, 2], min=0.0,  max=1.0).item()
                    MemoryActions.append((steer_val, accel_val, BrakeVal))

                for i, con in enumerate(ConList):
                    if not BatchDones[i]:
                        try:
                            con.send(MemoryActions[i])
                        except (EOFError, BrokenPipeError, OSError):
                            BatchDones[i] = True

                NextStates      = list(BatchStates)
                PreviousRewards = [0.0] * len(ConList)
                PreviousDones   = list(BatchDones)

                for i, con in enumerate(ConList):
                    if not BatchDones[i]:
                        try:
                            np_obs, reward, RaceDone = con.recv()
                            NextStates[i]      = np_obs
                            PreviousRewards[i] = float(reward)
                            PreviousDones[i]   = bool(RaceDone)
                        except (EOFError, BrokenPipeError, OSError):
                            PreviousDones[i] = True

                total_episode_reward += sum(PreviousRewards)
                BatchStates = NextStates
                BatchDones  = PreviousDones

                if (step + 1) % 100 == 0:
                    dist_pct = float(BatchStates[0][12]) * 100.0
                    steer_now = MemoryActions[0][0]
                    accel_now = MemoryActions[0][1]
                    print(f'  Step {step + 1:4d}/{MAX_STEPS} | Track Progress: {dist_pct:5.1f}% | Steer: {steer_now:+.2f} | Accel: {accel_now:.2f}')

                if all(PreviousDones):
                    print(f'\nRace finished at step {step + 1}!')
                    break

            print(f'Episode: {episode + 1}/{EpochLimit} | Total Reward: {total_episode_reward:.2f}')

    finally:
        for con in ConList:
            try:
                con.send('TERMINATE')
            except Exception:
                pass
        for process in ProcessList:
            process.join(timeout=5)
            if process.is_alive():
                process.terminate()


In [5]:
if __name__ == '__main__':
    main()


=== Starting Visual Run at Normal Real-Time Speed (50 FPS) ===
  Step  100/1500 | Track Progress:   8.8% | Steer: -0.99 | Accel: 1.00
  Step  200/1500 | Track Progress:  35.7% | Steer: +0.13 | Accel: 1.00
  Step  300/1500 | Track Progress:  62.9% | Steer: -0.27 | Accel: 1.00
  Step  400/1500 | Track Progress:  90.3% | Steer: +0.95 | Accel: 1.00

Race finished at step 434!
Episode: 1/1 | Total Reward: 68523.04
